# 01 — Data Preparation and Statistical Screening

Exploratory analysis of the coffee-market universe and the statistical
prerequisites for pairs trading:

1. load and clean the close-price dataset;
2. chronological in-sample / out-of-sample split (70/30, no look-ahead);
3. ADF unit-root tests — candidates must be non-stationary, i.e. I(1);
4. Engle-Granger cointegration tests against the Arabica anchor `KC1`;
5. selection of the 4-pair equity basket.

> **Data note:** the report figures were produced with Bloomberg data
> (not redistributable). Run `python scripts/download_data.py` first to
> reproduce this notebook with free Yahoo Finance data — see `data/README.md`
> for the differences between the two sources.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from src import backtest, data, hedging, metrics, signals, stat_tests

cfg = yaml.safe_load(open("../configs/default.yaml", encoding="utf-8"))
ANCHOR = cfg["universe"]["anchor"]
EQUITIES = cfg["universe"]["equities"]
WINDOW = cfg["signals"]["zscore_window"]

## 1. Load prices and inspect data quality

In [ ]:
prices = data.load_prices("../" + cfg["data"]["path"])
display(prices.tail())
display(data.data_quality_report(prices).head(10))

NaNs are concentrated in late-listed names (e.g. `BROS`, `WEST`, `THCH` in the
Bloomberg universe). Rather than dropping rows globally — which would shrink
the sample for every asset — those equities are excluded from the candidate
universe via the config.

## 2. Arabica vs. the universe — a first look

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(prices.index, prices[ANCHOR], color="saddlebrown", lw=1.2, label=ANCHOR)
ax.set_ylabel("Price (US cents/lb)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
ax.set_title("Arabica front-month future — the strategy anchor")
plt.tight_layout()
plt.show()

## 3. Log-prices and in-sample / out-of-sample split

In [ ]:
log_prices = data.to_log_prices(prices)
log_is, log_oos, split_date = data.train_test_split(
    log_prices, ANCHOR, cfg["split"]["ratio"]
)
print(f"In-sample : {log_is.index[0].date()} -> {split_date.date()} ({len(log_is)} obs)")
print(f"Out-sample: {split_date.date()} -> {log_oos.index[-1].date()} ({len(log_oos)} obs)")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(log_prices[ANCHOR], color="saddlebrown", lw=1.2)
ax.axvline(split_date, color="red", ls="--", lw=1.2, label="IS / OOS split")
ax.set_ylabel("log price")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

All screening and calibration below uses the **in-sample window only**.
The out-of-sample window stays untouched until the backtest in notebook 02.

## 4. ADF unit-root screening

In [ ]:
exclude = set(cfg["universe"]["exclude"]) | {ANCHOR}
adf_results = stat_tests.adf_screen(log_is, exclude=exclude)
adf_results.round(4)

High p-values fail to reject the unit root: the series are I(1), as required.
A *stationary* price series would already be mean-reverting on its own and
admit no long-run equilibrium relationship to trade.

## 5. Engle-Granger cointegration against KC1

In [ ]:
eg_results = stat_tests.engle_granger_screen(log_is, ANCHOR, exclude=exclude)
display(eg_results.round(4))

summary = stat_tests.summary_table(log_is, ANCHOR, adf_results, eg_results)
summary.round(4)

Only `SJM` is cointegrated with `KC1` at the 5% level in the Bloomberg
sample; the other candidates show weak evidence (p-values 0.22–0.53). The
strategy therefore relies on the **multivariate basket** — the optimizer
searches for a synthetic combination that mean-reverts better than any
individual pair. This weak cointegration foundation is the key structural
limitation discussed in the report.

## 6. Visual check of the selected basket

In [ ]:
top4 = eg_results.head(4)["asset"].tolist()
fig, axes = plt.subplots(len(top4), 1, figsize=(12, 3.5 * len(top4)))
for ax, asset in zip(np.atleast_1d(axes), top4):
    pair = pd.concat([prices[ANCHOR], prices[asset]], axis=1).dropna()
    ax2 = ax.twinx()
    ax.plot(pair.index, pair[ANCHOR], color="tab:blue", lw=1.1, label=ANCHOR)
    ax2.plot(pair.index, pair[asset], color="tab:red", lw=1.1, label=asset)
    pval = eg_results.loc[eg_results["asset"] == asset, "p_value"].iloc[0]
    ax.set_title(f"{asset} vs {ANCHOR} (Engle-Granger p = {pval:.4f})")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()